
# EEG Dataset with ML Notebook Example


In [7]:
import os
import json
import numpy as np
import pandas as pd
import mne

# === 1. Path to your locally downloaded dataset ===
# Change this if your dataset lives somewhere else.
dataset_root = os.path.expanduser("~/openneuro_data/ds005555")

# Choose subject
sub = "sub-1"

# EEG folder for that subject
eeg_path = os.path.join(dataset_root, sub, "eeg")

print("EEG path:", eeg_path)
print("Files in eeg_path:", os.listdir(eeg_path))

# === 2. Build file paths ===
psg_file    = os.path.join(eeg_path, f"{sub}_task-Sleep_acq-psg_eeg.edf")
events_tsv  = os.path.join(eeg_path, f"{sub}_task-Sleep_acq-psg_events.tsv")
events_json = os.path.join(eeg_path, f"{sub}_task-Sleep_acq-psg_events.json")

print("\nPSG file:", psg_file)
print("Events TSV:", events_tsv)
print("Events JSON:", events_json)

# === 3. Load EEG and events ===
raw = mne.io.read_raw_edf(psg_file, preload=True)
print("\nRaw object:", raw)
print("Channels:", raw.ch_names)
print("Sampling frequency:", raw.info["sfreq"])

events_df = pd.read_csv(events_tsv, sep="\t")
print("\nEvents head:")
print(events_df.head())

with open(events_json) as f:
    events_meta = json.load(f)
print("\nEvents JSON keys:", events_meta.keys())


EEG path: /Users/justinbanh/openneuro_data/ds005555/sub-1/eeg
Files in eeg_path: ['sub-1_task-Sleep_acq-headband_events.tsv', 'sub-1_task-Sleep_acq-headband_events.json', 'sub-1_task-Sleep_acq-headband_channels.tsv', 'sub-1_task-Sleep_acq-psg_eeg.edf', 'sub-1_task-Sleep_acq-psg_events.tsv', 'sub-1_task-Sleep_acq-headband_eeg.edf', 'sub-1_task-Sleep_acq-headband_eeg.json', 'sub-1_task-Sleep_acq-psg_eeg.json', 'sub-1_task-Sleep_acq-psg_events.json', 'sub-1_task-Sleep_acq-psg_channels.tsv']

PSG file: /Users/justinbanh/openneuro_data/ds005555/sub-1/eeg/sub-1_task-Sleep_acq-psg_eeg.edf
Events TSV: /Users/justinbanh/openneuro_data/ds005555/sub-1/eeg/sub-1_task-Sleep_acq-psg_events.tsv
Events JSON: /Users/justinbanh/openneuro_data/ds005555/sub-1/eeg/sub-1_task-Sleep_acq-psg_events.json
Extracting EDF parameters from /Users/justinbanh/openneuro_data/ds005555/sub-1/eeg/sub-1_task-Sleep_acq-psg_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading

In [8]:
print("Event columns:\n", events_df.columns)

# Sleep stages column name (from the BOAS dataset)
LABEL_COL = "stage_hum"

print("\nUnique values in stage_hum (sleep stages):")
print(events_df[LABEL_COL].unique())


Event columns:
 Index(['onset', 'duration', 'begsample', 'endsample', 'offset', 'stage_hum',
       'stage_ai'],
      dtype='object')

Unique values in stage_hum (sleep stages):
[3 2 0 1 8 4]


In [10]:
from mne.time_frequency import psd_array_welch

# === 1. Define column names ===
ONSET_COL = "onset"
DUR_COL   = "duration"
LABEL_COL = "stage_hum"

print("Using columns:")
print("  ONSET_COL =", ONSET_COL)
print("  DUR_COL   =", DUR_COL)
print("  LABEL_COL =", LABEL_COL)

# Drop rows without sleep-stage labels
events_clean = events_df.dropna(subset=[LABEL_COL]).copy()

# Map sleep stages to integers
unique_labels = sorted(events_clean[LABEL_COL].unique())
label_to_int = {lab: idx for idx, lab in enumerate(unique_labels)}
events_clean["label_int"] = events_clean[LABEL_COL].map(label_to_int)

print("\nLabel mapping (stage -> int):")
print(label_to_int)

# === 2. Basic preprocessing ===
raw.filter(0.5, 30.0, fir_design="firwin")

sfreq = raw.info["sfreq"]
print("\nSampling frequency:", sfreq)

# === 3. Build MNE events array ===
event_list = []
labels = []

for _, row in events_clean.iterrows():
    onset_sec = float(row[ONSET_COL])
    sample_idx = int(onset_sec * sfreq)
    label_int = int(row["label_int"])
    event_list.append([sample_idx, 0, label_int])
    labels.append(label_int)

events = np.array(event_list, dtype=int)
labels = np.array(labels, dtype=int)

print("\nEvents shape:", events.shape)
print("Labels shape:", labels.shape)

# === 4. Create 30-second epochs ===
epoch_len = 30.0

epochs = mne.Epochs(
    raw,
    events,
    event_id=None,
    tmin=0.0,
    tmax=epoch_len,
    baseline=None,
    preload=True,
    on_missing="ignore",
)

print("\nEpochs object:", epochs)

# === 5. Extract bandpower features ===
X_features = []

for ep in epochs:  # ep: (n_channels, n_times)
    psd, freqs = psd_array_welch(
        ep,
        sfreq=raw.info["sfreq"],
        fmin=0.5,
        fmax=30.0,
        n_fft=1024
    )

    band_defs = {
        "delta": (0.5, 4),
        "theta": (4, 8),
        "alpha": (8, 12),
        "beta":  (12, 30),
    }

    feat_vec = []
    for (fmin, fmax) in band_defs.values():
        mask = (freqs >= fmin) & (freqs < fmax)
        bandpower = psd[:, mask].mean(axis=1)
        feat_vec.extend(bandpower.tolist())

    X_features.append(feat_vec)

X = np.array(X_features)
y = labels[: len(X)]

print("\nFeature matrix X shape:", X.shape)
print("Label vector y shape:", y.shape)


Using columns:
  ONSET_COL = onset
  DUR_COL   = duration
  LABEL_COL = stage_hum

Label mapping (stage -> int):
{np.int64(0): 0, np.int64(1): 1, np.int64(2): 2, np.int64(3): 3, np.int64(4): 4, np.int64(8): 5}
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1691 samples (6.605 s)


Sampling frequency: 256.0

Events shape: (915, 3)
Labels shape: (915,)
Not setting metadata
915 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 915 events and

[Parallel(n_jobs=1)]: Done  11 out of  11 | elapsed:    1.2s finished


0 bad epochs dropped

Epochs object: <Epochs | 915 events (all good), 0 – 30 s (baseline off), ~589.8 MB, data loaded,
 '0': 200
 '1': 57
 '2': 398
 '3': 172
 '4': 87
 '5': 1>
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effectiv

[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]

Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective wind

...

[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]

Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective wind

[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]

Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective window size : 4.000 (s)
Effective wind

[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]

In [12]:
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# === 0. Inspect class counts ===
counts = Counter(y)
print("Original class counts (by int label):", counts)

# Keep only classes that have at least 2 samples
keep_classes = [cls for cls, c in counts.items() if c >= 2]
print("Keeping classes:", keep_classes)

mask = np.isin(y, keep_classes)
X_use = X[mask]
y_use = y[mask]

print("Filtered X shape:", X_use.shape)
print("Filtered y shape:", y_use.shape)

# Optional: map int labels back to stage names, but filtered
inv_label_map = {v: k for k, v in label_to_int.items()}
kept_label_names = [inv_label_map[i] for i in sorted(keep_classes)]
print("Kept label names:", kept_label_names)

# === 1. Train/test split (now safe to stratify) ===
X_train, X_test, y_train, y_test = train_test_split(
    X_use,
    y_use,
    test_size=0.2,
    random_state=42,
    stratify=y_use,
)

# === 2. Define and train the Random Forest ===
clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=42,
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# === 3. Evaluation ===
print("\nClassification report (filtered classes):")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[str(name) for name in kept_label_names]
    )
)

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))


Original class counts (by int label): Counter({np.int64(2): 398, np.int64(0): 200, np.int64(3): 172, np.int64(4): 87, np.int64(1): 57, np.int64(5): 1})
Keeping classes: [np.int64(3), np.int64(2), np.int64(0), np.int64(1), np.int64(4)]
Filtered X shape: (914, 44)
Filtered y shape: (914,)
Kept label names: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Classification report (filtered classes):
              precision    recall  f1-score   support

           0       0.97      0.90      0.94        40
           1       0.54      0.64      0.58        11
           2       0.88      0.96      0.92        80
           3       1.00      0.91      0.96        35
           4       0.92      0.71      0.80        17

    accuracy                           0.90       183
   macro avg       0.86      0.82      0.84       183
weighted avg       0.90      0.90      0.90       183

Confusion matrix:
[[36  3  1  0  0]
 [ 1  7  2  0  1]
 [ 0  3 77  0  0]
 [ 0  0  3 32  0]
 [ 0  